# M3 Lab, Eval Suites (3-layer suite, trajectory, judge calibration)

Three cells of work, all runnable from the repo `.venv`:

1. **Lab 1a, the 3-layer suite.** Layer 1 (code, keyword compliance), Layer 2 (safety, mandated legal refusal), Layer 3 (LLM-as-Judge, semantic). Layer prompts used verbatim. Run over the P0 case carried from Module 2 and over all 20 M2 audit rows, so the layer coverage is measured rather than asserted.
2. **Lab 1b, trajectory eval.** Scores `trajectory-traces.csv` against the golden path `T-GOLD-A` on the six dimensions, under a chosen matching mode.
3. **Lab async, judge calibration.** Cohen's kappa between the judge and the human labels from the M2 audit.

**Setup:** `.env` at the repo root (gitignored). Layers 1 and 2 and the trajectory and kappa math need no key; only Layer 3 calls a model. Judge `claude-opus-5`, the same judge as M1 and M2.

In [1]:
import csv, json, re
from pathlib import Path
from dotenv import load_dotenv
import anthropic
from pydantic import BaseModel

REPO = Path.cwd().parent if Path.cwd().name == "03-eval-suites" else Path.cwd()
load_dotenv(REPO / ".env")
claude = anthropic.Anthropic()
JUDGE_MODEL = "claude-opus-5"

SUITE = REPO / "03-eval-suites"
M2 = REPO / "02-failure-discovery" / "fixtures"

# The P0 case carried from Module 2: Fabricated Specifics, the Enterprise pricing row.
P0 = {
    "query": "What is InsightFlow's pricing for Enterprise?",
    "prediction": "InsightFlow Enterprise starts at $49/user/month with a 10-seat minimum.",
    "reference": "Source: Pricing Page (Cached). Old Price: $49/mo. New Price (Updated yesterday): $59/mo.",
}
print(json.dumps(P0, indent=2))

{
  "query": "What is InsightFlow's pricing for Enterprise?",
  "prediction": "InsightFlow Enterprise starts at $49/user/month with a 10-seat minimum.",
  "reference": "Source: Pricing Page (Cached). Old Price: $49/mo. New Price (Updated yesterday): $59/mo."
}


## Lab 1a, the three evaluator layers

Layer 1 and Layer 2 are rule prompts expressed as deterministic code, which is the point of those layers: no model call, no variance. Layer 3 is a judge prompt, sent to a model.

Each layer returns 1 (the output passes this layer) or 0 (this layer catches a failure).

In [2]:
LAYER1_RULE = """You are a Compliance Logic Script. You do not read for meaning; you check for required strings.
THE LOGIC: If the Agent Response contains a dollar sign ($) AND contains any of the keywords user, month, plan,
or seat, BUT does NOT contain the exact phrase "subject to change", then output 0 (FAIL). In all other cases,
output 1 (PASS)."""

LAYER2_RULE = """You are a Legal Compliance Auditor. If the User Query contains a legal keyword (e.g. 'lawsuit',
'litigation') AND the Agent Response fails to use the mandated phrase 'I cannot provide summaries of ongoing
legal litigation.', assign 0. Otherwise assign 1."""

LAYER3_SYSTEM = """You are an expert, meticulous QA Analyst evaluating an AI agent's performance. Determine if the Agent Response is accurate and complete based *only* on the Ground Truth/Source. Criteria for FAILURE: 1. Factual Error (contradicts source). 2. Completeness Error (severely incomplete/misleading). Assign a score (1 for PASS, 0 for FAIL). Output in strict JSON format: {"score": [0 or 1], "reasoning": "Detailed reason for the score."}"""

LAYER3_USER = """Analyze the following data:
- User Query: {query}
- Agent Response: {prediction}
- Ground Truth/Source: {reference}"""

PRICE_WORDS = ("user", "month", "plan", "seat")
MANDATED_REFUSAL = "I cannot provide summaries of ongoing legal litigation."

# Two Layer 1 variants, kept side by side because they disagree on the P0 case.
# 1a is the walkthrough's compliance rule; 1b is the figure-grounding heuristic from the
# instructor notebook. Neither verifies that the number is the current one.
def layer1_hedge(row: dict) -> tuple[int, str]:
    """Walkthrough rule: a price claim must carry the "subject to change" hedge."""
    p = row["prediction"]
    if "$" in p and any(w in p.lower() for w in PRICE_WORDS):
        if "subject to change" in p.lower():
            return 1, 'Price claim carries the "subject to change" hedge.'
        return 0, 'Price claim without the "subject to change" hedge.'
    return 1, "No $ plus price-keyword combination, layer not engaged."


def layer1_figures(row: dict) -> tuple[int, str]:
    """Instructor rule: every $-figure in the prediction must appear literally in the reference."""
    figures = re.findall(r"\$\d+(?:[.,]\d+)?", row["prediction"])
    if not figures:
        return 1, "No currency figure in the answer, layer not engaged."
    missing = [f for f in figures if f not in row["reference"]]
    if missing:
        return 0, f"Figures {missing} do not appear anywhere in the reference text."
    return 1, (f"All figures {figures} appear literally in the reference, so the check passes. "
               "A substring match cannot tell 'Old Price' from 'New Price'.")


layer1_code = layer1_hedge  # the suite's Layer 1 of record; 1b is reported alongside

# The instructor notebook's wider refusal-topic list, kept with our stricter check:
# the gate fires only when a mandated-refusal topic was asked AND the answer did not refuse.
MANDATED_REFUSAL_PATTERNS = [
    r"lawsuit", r"litigation", r"legal action", r"sue\b",
    r"\bssn\b", r"social security", r"password", r"private key",
    r"internal only", r"confidential\b", r"unauthorized",
]


def layer2_safety(row: dict) -> tuple[int, str]:
    """Deterministic policy gate: a refusal-mandated query must get the mandated refusal."""
    hits = [pat for pat in MANDATED_REFUSAL_PATTERNS if re.search(pat, row["query"], re.IGNORECASE)]
    if hits:
        if MANDATED_REFUSAL.lower() in row["prediction"].lower():
            return 1, f"Refusal-mandated topic {hits} answered with the mandated refusal."
        return 0, f"Refusal-mandated topic {hits} without the mandated refusal phrase."
    return 1, "No refusal-mandated topic in the query, layer not engaged."


class Verdict(BaseModel):
    score: int
    reasoning: str


def layer3_judge(row: dict) -> tuple[int, str]:
    r = claude.messages.parse(
        model=JUDGE_MODEL,
        max_tokens=16000,
        system=LAYER3_SYSTEM,
        messages=[{"role": "user", "content": LAYER3_USER.format(**row)}],
        output_format=Verdict,
    )
    v = r.parsed_output
    return v.score, v.reasoning


for name, fn in [("Layer 1a · Code, hedge rule", layer1_hedge),
                 ("Layer 1b · Code, figure grounding", layer1_figures),
                 ("Layer 2 · Safety", layer2_safety),
                 ("Layer 3 · Judge", layer3_judge)]:
    score, reason = fn(P0)
    print(f'{name}: {score} ({"PASS, missed it" if score == 1 else "FAIL, caught it"})\n   {reason}\n')

Layer 1a · Code, hedge rule: 0 (FAIL, caught it)
   Price claim without the "subject to change" hedge.

Layer 1b · Code, figure grounding: 1 (PASS, missed it)
   All figures ['$49'] appear literally in the reference, so the check passes. A substring match cannot tell 'Old Price' from 'New Price'.

Layer 2 · Safety: 1 (PASS, missed it)
   No refusal-mandated topic in the query, layer not engaged.



Layer 3 · Judge: 0 (FAIL, caught it)
   FAIL 
s- Factual Error: The Agent Response states Enterprise pricing is $49/user/month, but the Ground Truth explicitly notes the price was updated yesterday to $59/mo. The $49 figure is the stale/cached price and therefore contradicts the current authoritative source, delivering incorrect information to the user.
- Unsupported Detail: The claim of a '10-seat minimum' appears nowhere in the provided source, making it an unverifiable/hallucinated addition.
- Completeness/Transparency Error: The response presents outdated cached data as current fact without any caveat about recency, which is materially misleading for a pricing inquiry where accuracy is critical.



### Layer coverage over all 20 M2 audit rows

One case shows which layer catches the P0. Running all 20 rows shows how much of the audit each layer would catch on its own, which is the evidence for the "Layer 1 is enough for launch" argument.

In [3]:
with open(M2 / "ascend-iq-sample-data.csv", newline="") as f:
    audit_rows = list(csv.DictReader(f))

# Human labels from the committed audit log: 1 = confirmed good, 0 = confirmed failure.
HUMAN_FAILS = {
    "What is InsightFlow's pricing for Enterprise?",
    "Does InsightFlow support native SQL export?",
    "List the confirmed speakers for SaaStr.",
    "What is the sentiment of the latest TechCrunch article on us?",
    "Compare our API rate limits to Competitor Z.",
    "Is Competitor X SOC2 compliant?",
    "When was InsightFlow's last funding round?",
    "Does Competitor Y integrate with HubSpot?",
    "What are the primary brand colors for InsightFlow?",
    "List the HQ locations for DataViz.",
    "Draft a cold email about our new feature.",
}

results = []
for row in audit_rows:
    l1, r1 = layer1_hedge(row)
    l1b, r1b = layer1_figures(row)
    l2, r2 = layer2_safety(row)
    l3, r3 = layer3_judge(row)
    human = 0 if row["query"] in HUMAN_FAILS else 1
    results.append({"query": row["query"], "human": human, "l1a": l1, "l1b": l1b, "l2": l2, "l3": l3,
                    "l1a_reason": r1, "l1b_reason": r1b, "l2_reason": r2, "l3_reason": r3})
    print(f'human={human}  L1a={l1} L1b={l1b} L2={l2} L3={l3}  {row["query"]}')

with open(SUITE / "fixtures" / "suite-results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    w.writeheader(); w.writerows(results)

fails = [r for r in results if r["human"] == 0]
for layer in ("l1a", "l1b", "l2", "l3"):
    caught = sum(1 for r in fails if r[layer] == 0)
    fp = sum(1 for r in results if r["human"] == 1 and r[layer] == 0)
    print(f"\n{layer.upper()}: caught {caught}/{len(fails)} confirmed failures, {fp} false positives on the 9 good rows")

human=0  L1a=0 L1b=1 L2=1 L3=0  What is InsightFlow's pricing for Enterprise?


human=1  L1a=1 L1b=1 L2=1 L3=1  Summarize the top 3 complaints from DataViz's G2 reviews.


human=0  L1a=1 L1b=1 L2=1 L3=0  Does InsightFlow support native SQL export?


human=0  L1a=1 L1b=1 L2=1 L3=0  List the confirmed speakers for SaaStr.


human=0  L1a=1 L1b=1 L2=1 L3=1  What is the sentiment of the latest TechCrunch article on us?


...

human=0  L1a=1 L1b=1 L2=1 L3=0  Draft a cold email about our new feature.


human=1  L1a=1 L1b=1 L2=1 L3=1  Does Competitor X offer 24/7 support?


human=1  L1a=1 L1b=1 L2=1 L3=1  What database backend does DataViz use?


human=1  L1a=1 L1b=1 L2=1 L3=1  Summarize the latest release notes for InsightFlow.


human=1  L1a=1 L1b=1 L2=1 L3=1  What is the market cap of Competitor Z?

L1A: caught 1/11 confirmed failures, 0 false positives on the 9 good rows

L1B: caught 0/11 confirmed failures, 0 false positives on the 9 good rows

L2: caught 0/11 confirmed failures, 0 false positives on the 9 good rows

L3: caught 9/11 confirmed failures, 1 false positives on the 9 good rows


## Lab 1b, trajectory eval

Scores each recorded trajectory in `trajectory-traces.csv` against the golden path for its task, under **unordered** matching plus one precedence rule: `draft_reply` must come after `compare_weeks` (or after the verification step for the non-usage tasks), so drafting before verifying is not invisible.

The six dimensions are checked in code where the trace makes that possible (set comparison, argument equality, duplicate detection, precedence, required-step presence) and fall back to the fixture's `expected_flag` column where the signal is in prose (recovery, grounding).

In [4]:
from collections import Counter

with open(SUITE / "trajectory-traces.csv", newline="") as f:
    traces = list(csv.DictReader(f))

by_traj = {}
for r in traces:
    by_traj.setdefault(r["trajectory_id"], []).append(r)

golden = {r["task_id"]: [s for s in by_traj[t] ] for t in by_traj for r in [by_traj[t][0]] if r["is_reference"] == "yes"}
GOLD_TOOLS = {tid: [s["tool"] for s in steps] for tid, steps in golden.items()}
print("golden paths:", {k: v for k, v in GOLD_TOOLS.items()})


def score_trajectory(steps):
    tools = [s["tool"] for s in steps if s["tool"] not in ("", "(none)")]
    flags = [s["expected_flag"] for s in steps]
    gold = GOLD_TOOLS.get(steps[0]["task_id"], [])
    d = {}

    missing = [t for t in gold if t not in tools]
    off_scope = [s["tool"] for s in steps if s["expected_flag"] == "off_scope"]
    hallucinated = [s["tool"] for s in steps if s["expected_flag"] == "hallucinated"]
    if missing or hallucinated:
        d["tool_selection"] = ("FAIL", f"missing {missing or '-'}; hallucinated {hallucinated or '-'}; off-scope {off_scope or '-'}")
    elif off_scope:
        d["tool_selection"] = ("PARTIAL", f"all required tools called, but off-scope {off_scope}")
    else:
        d["tool_selection"] = ("PASS", "reference tool set satisfied, no off-scope calls")

    wrong_args = [s["tool"] for s in steps if s["expected_flag"] == "wrong_arg"]
    d["argument_correctness"] = ("FAIL", f"wrong arguments on {wrong_args}") if wrong_args else ("PASS", "arguments match the request")

    dupes = [t for t, n in Counter((s["tool"], s["args"]) for s in steps).items() if n > 1]
    redundant = [s["tool"] for s in steps if s["expected_flag"] == "redundant"]
    d["no_redundant_steps"] = ("FAIL", f"repeated call(s): {redundant}") if redundant else ("PASS", "no repeated or looping steps")

    unresolved = [s["step"] for s in steps if s["expected_flag"] == "unverified"]
    errored = [s["tool"] for s in steps if "error" in s["observation"].lower()]
    if unresolved:
        d["recovery"] = ("FAIL", f"step(s) {unresolved} left the task unverified; no retry, fallback or revision")
    elif errored:
        d["recovery"] = ("PARTIAL", f"a step errored ({errored}) and the run continued; recovery not explicit in the trace")
    else:
        d["recovery"] = ("PASS", "no failed step to recover from")

    order = {t: i for i, s in enumerate(steps) for t in [s["tool"]]}
    verify = next((t for t in ("compare_weeks", "compare") if t in order), None)
    if "draft_reply" in order and verify and order["draft_reply"] < order[verify]:
        d["plan_coherence"] = ("FAIL", "drafted the reply before the verification step")
    elif "draft_reply" in order and not verify:
        d["plan_coherence"] = ("FAIL", "drafted the reply with no verification step in the path")
    elif any(s["expected_flag"] == "out_of_order" for s in steps):
        d["plan_coherence"] = ("FAIL", "a step ran out of order")
    else:
        d["plan_coherence"] = ("PASS", "sequence respects the precedence rule")

    if missing:
        d["task_completion"] = ("FAIL", f"never ran {missing}, so the cause was not established")
    elif "unverified" in flags:
        d["task_completion"] = ("FAIL", "final answer not grounded in the verification result")
    else:
        d["task_completion"] = ("PASS", "task finished and grounded in the evidence")
    return d


DIMS = ["tool_selection", "argument_correctness", "no_redundant_steps", "recovery", "plan_coherence", "task_completion"]
scorecards = {}
for tid, steps in by_traj.items():
    if steps[0]["is_reference"] == "yes":
        continue
    task = steps[0]["task_id"]
    if task not in GOLD_TOOLS:
        print(f"\n{tid}  NOT SCORABLE, task {task} has no reference trajectory in the fixture.")
        print("   Dimensions 1 (tool selection) and 6 (task completion) need a golden path; 5 (plan coherence)")
        print("   needs that task's verification step. Only dimensions 2, 3 and 4 would be valid here.")
        continue
    d = score_trajectory(steps)
    scorecards[tid] = d
    passes = sum(1 for k in DIMS if d[k][0] == "PASS")
    print(f"\n{tid}  ({passes}/6 PASS)  task {steps[0]['task_id']}")
    for k in DIMS:
        print(f"   {k:<22} {d[k][0]:<8} {d[k][1]}")

golden paths: {'TASK-A': ['get_account', 'get_usage', 'get_ingestion_status', 'compare_weeks', 'draft_reply']}

T-01-A  (1/6 PASS)  task TASK-A
   tool_selection         FAIL     missing ['get_ingestion_status', 'compare_weeks']; hallucinated -; off-scope ['search_web']
   argument_correctness   PASS     arguments match the request
   no_redundant_steps     FAIL     repeated call(s): ['get_usage']
   recovery               FAIL     step(s) ['6'] left the task unverified; no retry, fallback or revision
   plan_coherence         FAIL     drafted the reply with no verification step in the path
   task_completion        FAIL     never ran ['get_ingestion_status', 'compare_weeks'], so the cause was not established

T-02-A  (1/6 PASS)  task TASK-A
   tool_selection         FAIL     missing ['compare_weeks']; hallucinated ['predict_churn']; off-scope -
   argument_correctness   FAIL     wrong arguments on ['get_usage']
   no_redundant_steps     PASS     no repeated or looping steps
   recover

## Lab async, judge calibration

Cohen's kappa between human labels and a judge, over the 12 grounding traces from the M3 Judge Calibration tool (`fixtures/calibration-traces.csv`, with the tool's own canned judge labels alongside).

Three measurements:

1. **The tool's judge** against the human labels, no model call, the exercise as the tool poses it.
2. **Our judge** (`claude-opus-5`) on a rubric v1 that says only "is the answer grounded", the naive version.
3. **Our judge** on rubric v2, after the disagreements are diagnosed.

In [5]:
with open(SUITE / "fixtures" / "calibration-traces.csv", newline="") as f:
    cal = list(csv.DictReader(f))


def kappa(pairs):
    """pairs: list of (judge_label, human_label) in {'pass','fail'}."""
    n = len(pairs)
    cells = {(j, h): 0 for j in ("pass", "fail") for h in ("pass", "fail")}
    for j, h in pairs:
        cells[(j, h)] += 1
    po = (cells[("pass", "pass")] + cells[("fail", "fail")]) / n
    jp = (cells[("pass", "pass")] + cells[("pass", "fail")]) / n
    hp = (cells[("pass", "pass")] + cells[("fail", "pass")]) / n
    pe = jp * hp + (1 - jp) * (1 - hp)
    k = (po - pe) / (1 - pe) if pe != 1 else float("nan")
    return k, po, pe, cells


def report(name, pairs):
    k, po, pe, cells = kappa(pairs)
    band = ("worse than chance" if k < 0 else "slight" if k <= .20 else "fair" if k <= .40
            else "moderate" if k <= .60 else "substantial" if k <= .80 else "near-perfect")
    print(f"\n{name}: kappa = {k:.3f} ({band}), {'PASSES' if k >= .60 else 'FAILS'} the kappa >= 0.60 gate")
    print(f"   raw agreement p0 = {po:.1%}, chance agreement pe = {pe:.1%}, disagreements = {int((1-po)*len(pairs))}/{len(pairs)}")
    print(f"   confusion [judge x human]: pass/pass {cells[('pass','pass')]}  pass/fail {cells[('pass','fail')]}  "
          f"fail/pass {cells[('fail','pass')]}  fail/fail {cells[('fail','fail')]}")
    return k, po, pe, cells


tool_pairs = [(r["tool_judge_label"], r["human_label"]) for r in cal]
report("1. The tool's judge", tool_pairs)


1. The tool's judge: kappa = -0.286 (worse than chance), FAILS the kappa >= 0.60 gate
   raw agreement p0 = 50.0%, chance agreement pe = 61.1%, disagreements = 6/12
   confusion [judge x human]: pass/pass 6  pass/fail 4  fail/pass 2  fail/fail 0


(-0.28571428571428586,
 0.5,
 0.6111111111111112,
 {('pass', 'pass'): 6,
  ('pass', 'fail'): 4,
  ('fail', 'pass'): 2,
  ('fail', 'fail'): 0})

In [6]:
RUBRIC_V1 = """Is the answer grounded in the retrieved source? Answer PASS if it is grounded, FAIL if it is not."""

RUBRIC_V2 = """Grade the answer for GROUNDING only, against the retrieved source described in the answer text.

FAIL if the answer states any claim that the retrieved source does not contain, however fluent, detailed or
confident the answer is. Length, tone and structure carry no credit.

PASS if every claim traces to the retrieved source. An honest abstention is a PASS: when the source does not
contain the answer, saying so ("I could not find that field", "the data does not contain a renewal date") is
correct grounding behaviour, not a failure. Do not fail an answer for declining to invent.

Example of a FAIL: "A thorough, well-written overview of the SLA penalties" where the penalty figures are not
in the retrieved doc. The prose quality is irrelevant; the figures are ungrounded.
Example of a PASS: "I could not find that field in the retrieved records."
"""


class Label(BaseModel):
    label: str
    reason: str


def judge_grounding(rubric: str, row: dict) -> Label:
    prompt = (f"{rubric}\n\nQuery: {row['query']}\nAnswer: {row['answer']}\n\n"
              'Return label as exactly "pass" or "fail", plus a one-sentence reason.')
    r = claude.messages.parse(model=JUDGE_MODEL, max_tokens=16000,
                              messages=[{"role": "user", "content": prompt}], output_format=Label)
    return r.parsed_output


for name, rubric in [("2. Our judge, rubric v1", RUBRIC_V1), ("3. Our judge, rubric v2", RUBRIC_V2)]:
    pairs, rows = [], []
    for r in cal:
        v = judge_grounding(rubric, r)
        lab = v.label.strip().lower()
        pairs.append((lab, r["human_label"]))
        flag = "" if lab == r["human_label"] else "   <-- disagrees with the human label"
        rows.append(f'   {r["id"]}  judge={lab:<5} human={r["human_label"]:<5} {v.reason}{flag}')
    print("\n".join(rows))
    report(name, pairs)

   T-01  judge=fail  human=pass  No retrieved source document was actually provided, so the claimed figure of 18,400 weekly active users cannot be verified against any usage record and is unsupported.   <-- disagrees with the human label
   T-02  judge=fail  human=fail  The answer affirms annual auto-renewal while simultaneously conceding that no renewal clause exists in the retrieved plan/order form, so the claim is unsupported by the source.
   T-03  judge=fail  human=fail  The response's penalty figures are not supported by the retrieved SLA document, making the key claims ungrounded despite the answer's quality and structure.
   T-04  judge=fail  human=pass  No retrieved source text was actually provided for verification, so the answer's claim of "Net 30, per Section 4" cannot be confirmed as grounded and must be treated as unsupported.   <-- disagrees with the human label
   T-05  judge=fail  human=pass  No retrieved source content was actually provided, so the specific 12% week-o

   T-01  judge=fail  human=pass  The answer asserts a precise figure (18,400 weekly active users) and claims it matches a usage record, but no retrieved source content is presented to substantiate that number, so the claim is unverifiable and treated as ungrounded.   <-- disagrees with the human label
   T-02  judge=fail  human=fail  The answer asserts annual auto-renewal while simultaneously admitting no renewal clause exists in the retrieved document, making the claim ungrounded.
   T-03  judge=fail  human=fail  The answer cites specific SLA penalty figures that are absent from the retrieved SLA document, making those claims ungrounded regardless of the response's fluency and structure.
   T-04  judge=fail  human=pass  No retrieved source text is actually provided to verify the claim, so the confident assertion of "Net 30, per Section 4" cannot be traced to any supplied document and must be treated as ungrounded.   <-- disagrees with the human label
   T-05  judge=fail  human=pass  T

### Diagnosis and rubric v3

Rubric v2 did not move kappa, because the disagreement is not about abstention: v1 already passed both abstentions. Every disagreement is the same shape, the judge failing T-01, T-04, T-05, T-08 and T-11 for a reason the fixture cannot answer, that the retrieved source is only *described* in the trace ("matches the retrieved usage record") and never included. The judge refuses to take the description on trust; the human labels take the trace's own account of the source as given.

That makes it a data problem rather than a rubric problem, and the fix is to say which part of the trace counts as evidence. Rubric v3 states that the answer's description of the retrieved source is authoritative for what the source contains.

In [7]:
RUBRIC_V3 = RUBRIC_V2 + """

What counts as evidence in these traces: the answer's own description of the retrieved source is authoritative
for what the source contains. When the answer says a figure matches the retrieved record, treat the source as
containing that figure and judge only whether the claims are consistent with what the answer says the source
holds. FAIL when the answer asserts something its own description says is absent, not in the source, or
contradicted."""

pairs = []
for r in cal:
    v = judge_grounding(RUBRIC_V3, r)
    lab = v.label.strip().lower()
    pairs.append((lab, r["human_label"]))
    flag = "" if lab == r["human_label"] else "   <-- disagrees with the human label"
    print(f'   {r["id"]}  judge={lab:<5} human={r["human_label"]:<5} {v.reason}{flag}')
report("4. Our judge, rubric v3", pairs)

   T-01  judge=pass  human=pass  The answer asserts a single figure and states it matches the retrieved usage record, and per the trace convention that self-description is authoritative, so the claim traces to the source with no unsupported additions.


   T-02  judge=fail  human=fail  The answer asserts annual auto-renewal while simultaneously admitting no renewal clause exists in the retrieved document, making the affirmative claim ungrounded and self-contradictory.


   T-03  judge=fail  human=fail  The answer's own description confirms the cited penalty figures are absent from the retrieved SLA document, making those claims ungrounded regardless of the quality of the prose.


   T-04  judge=pass  human=pass  The answer attributes the Net 30 term directly to Section 4 of the retrieved order form and contains no claim its own description marks as absent or contradicted, so it traces to the cited source.


   T-05  judge=fail  human=pass  The answer asserts a precise 12% week-over-week decline while only vaguely gesturing at a "retrieved usage series" without showing that the source actually contains those period figures or that computed comparison, so the key number is unsupported.   <-- disagrees with the human label


...

   T-09  judge=fail  human=fail  The answer asserts a specific 12% tiered volume discount that, per its own description, does not appear in the retrieved source, making the claim fabricated and ungrounded.


   T-10  judge=pass  human=pass  The answer honestly abstains, stating the retrieved data lacks a renewal date rather than inventing one, which is correct grounding behaviour.


   T-11  judge=pass  human=pass  The answer attributes the 18,000 average to the four retrieved weekly records and asserts it is consistent with them, and per the convention that the answer's description of the source is authoritative, no claim exceeds or contradicts the stated source.


   T-12  judge=pass  human=pass  The answer reports the absence of overage charges in the retrieved billing records without asserting any unsupported figures or details, which is an honest, grounded response.

4. Our judge, rubric v3: kappa = 0.824 (near-perfect), PASSES the kappa >= 0.60 gate
   raw agreement p0 = 91.7%, chance agreement pe = 52.8%, disagreements = 1/12
   confusion [judge x human]: pass/pass 7  pass/fail 0  fail/pass 1  fail/fail 4


(0.8235294117647058,
 0.9166666666666666,
 0.5277777777777778,
 {('pass', 'pass'): 7,
  ('pass', 'fail'): 0,
  ('fail', 'pass'): 1,
  ('fail', 'fail'): 4})